In [1]:
import numpy as np
import soundfile as sf
import os
from IPython.display import Audio 
from os.path import join as pjoin
import tqdm
import librosa as lsa
import torch
import pyrubberband as prb

import sys
sys.path.insert(0, '/home/enric.guso@local.eurecat.org/ssl-singer-identity')

from singer_identity import load_model
model = load_model('byol')
model.eval();

def cos_dis(a, b):
    similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))    
    return 1 - similarity

def augment_voice(bvoc, fs):
    # bvoc is 1D array with the original transformed backing vocal
    # fs is sampling rate
    jitter = 200
    
    # RANDOM AMPLITUDE ENVELOPE
    # Detectem els inicis de sons (onsets) que solen coincidir amb les síl·labes
    onset_frames = lsa.onset.onset_detect(y=bvoc, sr=fs, wait=1, pre_avg=1, post_avg=1, pre_max=1, post_max=1)
    onset_times = lsa.frames_to_time(onset_frames, sr=fs)
    onset_samps = lsa.frames_to_samples(onset_frames)

    #amp_env_val = (np.random.rand(len(onset_samps)) + 0.5)
    amp_env_val = 3 * np.random.rand(len(onset_samps))+0.3

    amp_env = list(np.linspace(1, amp_env_val[0], onset_samps[0])) # first value, from zero
    for i, env_val in enumerate(amp_env_val):
        if i<(len(amp_env_val)-1):
            amp_env += list(np.linspace(env_val, amp_env_val[i+1], onset_samps[i+1]-onset_samps[i]))
    amp_env += list(np.linspace(amp_env_val[-1], 1, len(bvoc)-onset_samps[-1]))
    amp_env = np.array(amp_env)
    out1 = bvoc * amp_env
    
    # TIME-STRETCHING
    time_map = []
    prev_offset = 0
    time_map.append((0, 0))
    for i, onset in enumerate(onset_samps):
        noise = np.random.randint(low=0, high=int(fs * jitter/1000))
        act_onset = onset + prev_offset
        act_onset += noise
        if i < len(onset_samps)-1:
            if act_onset > onset_samps[i+1] :
                prev_offset = act_onset - onset_samps[i+1]
            else:
                prev_offset = 0
        time_map.append((onset, act_onset))
    time_map.append((len(bvoc), len(bvoc)))
    return prb.pyrb.timemap_stretch(out1, fs, time_map)

/media/diskA/enric/venvs/singerID/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetch custom.py: Delegating to Huggingface hub, source BernardoTorres/singer-identity.
Model loaded from pretrained_models/IdentityEncoder-677243260058dc75b961703db2ea0590/byol/model.pt


/media/diskA/enric/venvs/singerID/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/media/diskA/enric/venvs/singerID/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [2]:
song_name = 'Andy Bennett - Baby Let Me Hold You Tonight'

adjust_list = os.listdir(pjoin('/media/diskA/enric/musdbmoises_voices_adjust', song_name))
noadjust_list = os.listdir(pjoin('/media/diskA/enric/musdbmoises_voices_noadjust', song_name))

adjust_list.sort()
noadjust_list.sort()

crowdioset_list = os.listdir('/media/diskA/enric/crowdioset/')

crowdioset_list.sort()

vocals, fs = sf.read(pjoin(pjoin('/media/diskA/enric/musdbmoises/train', song_name), 'vocals.wav'))
drums, fs = sf.read(pjoin(pjoin('/media/diskA/enric/musdbmoises/train', song_name), 'drums.wav'))
bass, fs = sf.read(pjoin(pjoin('/media/diskA/enric/musdbmoises/train', song_name), 'bass.wav'))
other, fs = sf.read(pjoin(pjoin('/media/diskA/enric/musdbmoises/train', song_name), 'other.wav'))

In [3]:
noadjust_voices = np.zeros((len(noadjust_list), vocals.shape[0]))
for i, voice in enumerate(tqdm.tqdm(noadjust_list)):
    _m, _ = sf.read(pjoin(pjoin('/media/diskA/enric/musdbmoises_voices_noadjust', song_name), voice))
    noadjust_voices[i,:] = np.concatenate((_m, np.zeros(vocals.shape[0]-_m.shape[0])))
print('All noadjust adudio loaded into RAM.')

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:22<00:00,  8.96it/s]

All noadjust adudio loaded into RAM.


In [4]:
adjust_voices = np.zeros_like(noadjust_voices)
for i, voice in enumerate(tqdm.tqdm(adjust_list)):
    _m, _ = sf.read(pjoin(pjoin('/media/diskA/enric/musdbmoises_voices_adjust', song_name), voice))
    adjust_voices[i, :] = np.concatenate((_m, np.zeros(vocals.shape[0]-_m.shape[0])))
print('All adjust adudio loaded into RAM.')

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:21<00:00,  9.10it/s]

All adjust adudio loaded into RAM.


In [5]:
#the original singer embedding
embedding = model(torch.from_numpy(vocals.mean(1)).unsqueeze(0).float()).detach().numpy()[0] 

In [6]:
embeddings_noadjust = []
for voice in tqdm.tqdm(noadjust_voices):
    embeddings_noadjust.append(model(torch.from_numpy(voice).unsqueeze(0).float()).detach().numpy())

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [03:37<00:00,  1.09s/it]


In [7]:
embeddings_adjust = []
for i, voice in enumerate(tqdm.tqdm(adjust_voices)):
    embeddings_adjust.append(model(torch.from_numpy(voice).unsqueeze(0).float()).detach().numpy())

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [03:31<00:00,  1.06s/it]


In [8]:
embeddings_noadjust = np.array(embeddings_noadjust)[:,0,:]
embeddings_adjust = np.array(embeddings_adjust)[:,0,:]

In [9]:
singer_dis_noadjust = []
singer_dis_adjust = []

for e in embeddings_noadjust:
    singer_dis_noadjust.append(cos_dis(e, embedding))
for e in embeddings_adjust:
    singer_dis_adjust.append(cos_dis(e, embedding))

In [10]:
singer_dis_noadjust = np.array(singer_dis_noadjust)
singer_dis_adjust = np.array(singer_dis_adjust)

In [11]:
singer_sim_noadjust = 1 - singer_dis_noadjust
singer_sim_adjust = 1 - singer_dis_adjust

In [13]:
cgram = lsa.feature.chroma_stft(y=vocals.mean(1), sr=fs)

In [14]:
chrome_dist_noadjust = []
for voice in tqdm.tqdm(noadjust_voices):
    chrome_dist_noadjust.append(np.mean(np.abs(lsa.feature.chroma_stft(y=voice, sr=fs) - cgram)))
chrome_dist_adjust = []
for voice in tqdm.tqdm(adjust_voices):
    chrome_dist_adjust.append(np.mean(np.abs(lsa.feature.chroma_stft(y=voice, sr=fs) - cgram)))

100%|████████████████████████████████████████████████████████████████████████████████████████| 200/200 [03:31<00:00,  1.06s/it]


In [15]:
nchrome_dist_noadjust = (chrome_dist_noadjust - np.mean(chrome_dist_noadjust))/np.std(chrome_dist_noadjust)
nchrome_dist_adjust = (chrome_dist_adjust - np.mean(chrome_dist_adjust))/np.std(chrome_dist_adjust)

nsims_noadjust = (singer_sim_noadjust - np.mean(singer_sim_noadjust))/np.std(singer_sim_noadjust)
nsims_adjust = (singer_sim_adjust - np.mean(singer_sim_adjust))/np.std(singer_sim_adjust)

In [18]:
# using the analysis above, we pick 50-best voices
filt_voices = np.vstack((adjust_voices[np.argsort(nchrome_dist_adjust + nsims_adjust)[:50]],
noadjust_voices[np.argsort(nchrome_dist_noadjust + nsims_noadjust)[:50]]))

In [20]:
augmented_voices = []
for voice in tqdm.tqdm(filt_voices):
    augmented_voices.append(augment_voice(voice, fs))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [10:30<00:00,  6.31s/it]


In [25]:
augmented_voices = np.array(augmented_voices)

In [27]:
np.save('augmented_voices.npy', augmented_voices)

In [21]:
# With noadjust almost all are chromatically, i'd just discard the last 10%
# With adjust they get out of tune very easily, i'd discard at least half of it. they also sound more different